In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset=pd.read_csv("insurance_pre.csv")

In [3]:
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
#Nominal - One-Hot-Encoding
#used function is "get.dummies" - the reason they used for this naming??
# when two field is zero automatically even if we don't mention newyork- newyork will get 1
#Column expansion
#Three uniq values such as california, florida, newyork
#reason for the column name expansion - State_California (state is the original column name)

In [5]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [6]:
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [7]:
#Now we are going to split the input and output dataset
#
#
#

In [8]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [9]:
independent=dataset[['age', 'bmi', 'children', 'sex_male', 'smoker_yes']]

In [10]:
independent 

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [11]:
dependent=dataset[["charges"]]

In [12]:
dependent

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [13]:
#split into training set and test test
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(independent, dependent,test_size=0.30,random_state=0)

In [14]:
X_train

,age,bmi,children,sex_male,smoker_yes
1163,18,28.215,0,0,0
196,39,32.800,0,0,0
438,52,46.750,5,0,0
183,44,26.410,0,0,0
1298,33,27.455,2,1,0
...,...,...,...,...,...
763,27,26.030,0,1,0
835,42,35.970,2,1,0
1216,40,25.080,0,1,0
559,19,35.530,0,1,0


In [15]:
#Standardization


In [16]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()

X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)

In [17]:
X_train

array([[-1.5330973 , -0.40713453, -0.89833872, -0.97676557, -0.50466988],
       [-0.03364163,  0.32855417, -0.89833872, -0.97676557, -0.50466988],
       [ 0.89459283,  2.56690911,  3.25603402, -0.97676557, -0.50466988],
       ...,
       [ 0.03776102, -0.91016269, -0.89833872,  1.02378711, -0.50466988],
       [-1.46169465,  0.76659782, -0.89833872,  1.02378711, -0.50466988],
       [-0.46205754, -1.96596021, -0.06746417, -0.97676557, -0.50466988]])

In [18]:
from sklearn.ensemble import RandomForestRegressor

In [27]:
from sklearn.model_selection import GridSearchCV
#from sklearn.tree import DecisionTreeRegressor
param_grid= {'criterion':['squared_error','absolute_error'], 'max_features':[1.0,'sqrt', 'log2'],'n_estimators':[10,100]}

grid= GridSearchCV(RandomForestRegressor(), param_grid, refit=True, verbose=3, n_jobs=-1)

#fitting model

grid.fit(X_train,y_train)


Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\Users\rousan\anaconda3\Lib\site-packages\sklearn\base.py:1151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


GridSearchCV(estimator=RandomForestRegressor(), n_jobs=-1,
             param_grid={'criterion': ['squared_error', 'absolute_error'],
                         'max_features': [1.0, 'sqrt', 'log2'],
                         'n_estimators': [10, 100]},
             verbose=3)

In [28]:
# print best parameter after tuning
# print (grid.best_params_)

re=grid.cv_results_

# print(re)

grid_predictions= grid.predict(X_test)

# print classification report

from sklearn.metrics import r2_score

r_score=r2_score(y_test,grid_predictions)

print("The R_score value for best parameter {}:" .format(grid.best_params_))

The R_score value for best parameter {'criterion': 'absolute_error', 'max_features': 'sqrt', 'n_estimators': 100}:


In [29]:
table=pd.DataFrame.from_dict(re)

In [30]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.169002,0.005879,0.013494,0.008030,squared_error,1.0,10,"{'criterion': 'squared_error', 'max_features':...",0.841819,0.761596,0.799371,0.814410,0.762823,0.796004,0.030770,8
1,1.623076,0.019462,0.039461,0.006370,squared_error,1.0,100,"{'criterion': 'squared_error', 'max_features':...",0.861659,0.766876,0.818519,0.794939,0.773411,0.803081,0.034430,5
2,0.122916,0.006566,0.005259,0.004328,squared_error,sqrt,10,"{'criterion': 'squared_error', 'max_features':...",0.845078,0.783959,0.787636,0.796357,0.744191,0.791444,0.032262,9
3,1.094617,0.011861,0.039507,0.001575,squared_error,sqrt,100,"{'criterion': 'squared_error', 'max_features':...",0.868426,0.789851,0.809435,0.826453,0.766521,0.812137,0.034520,3
4,0.115759,0.003710,0.011459,0.006404,squared_error,log2,10,"{'criterion': 'squared_error', 'max_features':...",0.833459,0.765728,0.794182,0.815072,0.748385,0.791365,0.031133,10
5,1.098605,0.010093,0.042414,0.006017,squared_error,log2,100,"{'criterion': 'squared_error', 'max_features':...",0.863722,0.795023,0.811560,0.831535,0.768527,0.814073,0.032291,2
6,0.439396,0.008564,0.012972,0.008128,absolute_error,1.0,10,"{'criterion': 'absolute_error', 'max_features'...",0.837879,0.745753,0.782891,0.775671,0.762225,0.780884,0.031183,12
7,3.425652,0.517684,0.019455,0.003959,absolute_error,1.0,100,"{'criterion': 'absolute_error', 'max_features'...",0.844997,0.781003,0.798420,0.796285,0.763388,0.796819,0.027173,7
8,0.261947,0.011634,0.012810,0.006411,absolute_error,sqrt,10,"{'criterion': 'absolute_error', 'max_features'...",0.849366,0.760447,0.800792,0.767140,0.731745,0.781898,0.040255,11
9,2.407707,0.321245,0.032841,0.010101,absolute_error,sqrt,100,"{'criterion': 'absolute_error', 'max_features'...",0.864708,0.805025,0.805856,0.824606,0.774814,0.815002,0.029533,1


In [31]:
age_input=float(input("Age:"))
bmi_input=float(input("BMI:"))
children_input=float(input("Children:"))
sex_male_input=int(input("Sex Male 0 or 1:"))
smoker_yes_input=int(input("Smoker Yes 0 or 1:"))


Age:43
BMI:21
Children:2
Sex Male 0 or 1:0
Smoker Yes 0 or 1:1


In [32]:
Future_Prediction=grid.predict([[age_input,bmi_input,children_input,sex_male_input,smoker_yes_input]])
print("Future_Prediction={}". format(Future_Prediction))

Future_Prediction=[47611.31349515]
